# htr-metrics-transfer: training-free metrics of the wave-1 augmentations

For every condition (`none` + geom/elastic/stroke/photo x levels 1-4) and draw k = 0..2 (`none`: 1 draw), the training set (Option-B train minus the 20% dev writers, the exact lines the recognizers saw) is augmented once with `augment.py` and scored against the un-augmented dev lines at 4 granularities (patch / word / line with DINOv2-small, vit_token with the TrOCR-base encoder): coverage, density, precision, Vendi (+ SD over 10 subsamples), mean pairwise distance, proxy A-distance, KID.

49 jobs, split over the 2 GPUs (one process per T4). Every finished job is pushed to `cyttic/hmt-metrics` (`per/<cond>_k<k>.json`), so **Save & Run All again resumes** after a timeout. Expected about 2-3 h.

**Setup:** Accelerator = GPU T4 x2, Internet = ON, secret `HF_TOKEN` (write).

In [ ]:
!pip install -q 'pillow==11.2.1'   # Kaggle's base Pillow breaks transformers; the workers are fresh processes, so no restart needed

In [ ]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    tok = None
login(token=tok) if tok else login()

In [ ]:
%%writefile augment.py
# -*- coding: utf-8 -*-
"""Augmentation families x strength levels for the augmentation / training-free-metrics study.

ONE source of truth: the training notebooks (online augmentation) and the metric computation (K augmented draws of the training
set) both call `augment()` from this file, so the metrics describe exactly the distribution the recognizer was trained on.

Input / output: a grayscale uint8 line image (H x W, white background, dark ink) ALREADY resized to the model line height
(64 px), so every magnitude below is in pixels of a 64 px high line. Level 0 = identity. For level L >= 1 every image is
augmented (p = 1); each magnitude is drawn uniformly from [0, MAX[L]] (with a random sign where it has one), so a higher
level widens the distribution rather than shifting it.

Families
  geom     rotation U(-a, a) deg + horizontal shear U(-s, s); the canvas is enlarged so no ink is cut, then resized back to 64 px
           high (a rotated long line gets taller, so its letters shrink: part of what rotating a cropped line does)
  elastic  smooth random displacement field (Gaussian-blurred noise, sigma 4 px), RMS displacement U(0, A) px per axis
  stroke   ink thickening with an elliptical kernel of size r+1, r ~ U{-1..k}; r = -1 thins by one px (more thinning erases
           HHD's ~3 px strokes), r = 0 leaves the line unchanged
  photo    Gaussian blur sigma U(0, b) + ink-contrast loss (ink lightened by up to c) + Gaussian pixel noise std U(0, n)
"""
import math

import cv2
import numpy as np

LEVELS = (0, 1, 2, 3, 4)
FAMILIES = ("geom", "elastic", "stroke", "photo")
MAX = {
    "geom":    {"rot": (0, 0.75, 1.5, 2.25, 3.0), "shear": (0, 0.10, 0.20, 0.35, 0.50)},
    "elastic": {"amp": (0, 0.75, 1.5, 2.25, 3.0), "sigma": 4.0},
    "stroke":  {"k": (0, 1, 2, 3, 4)},
    "photo":   {"blur": (0, 0.5, 1.0, 1.5, 2.0), "contrast": (0, 0.15, 0.30, 0.45, 0.60), "noise": (0, 0.03, 0.06, 0.10, 0.15)},
}


def condition_name(family, level):
    return "none" if level == 0 or family in (None, "none") else f"{family}{level}"


def parse_condition(name):
    """'none' -> (None, 0); 'geom3' -> ('geom', 3)"""
    if name == "none": return None, 0
    return name[:-1], int(name[-1])


def _geom(g, L, rng):
    a = math.radians(rng.uniform(-1, 1) * MAX["geom"]["rot"][L]); s = rng.uniform(-1, 1) * MAX["geom"]["shear"][L]
    h0, w0 = g.shape
    px = int(math.ceil(abs(s) * h0 + abs(math.sin(a)) * h0)) + 2; py = int(math.ceil(abs(math.sin(a)) * w0 / 2)) + 1
    g = cv2.copyMakeBorder(g, py, py, px, px, cv2.BORDER_CONSTANT, value=255); h, w = g.shape
    cx, cy = w / 2, h / 2
    R = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]]) @ np.array([[1, s], [0, 1]])
    M = np.hstack([R, (np.array([cx, cy]) - R @ np.array([cx, cy]))[:, None]])
    g = cv2.warpAffine(g, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=255)
    rows = np.where((g < 200).any(1))[0]                       # crop the vertical padding back to the ink (+ the original margin)
    if len(rows): g = g[max(0, rows[0] - 2):rows[-1] + 3]
    return cv2.resize(g, (max(1, round(g.shape[1] * h0 / g.shape[0])), h0), interpolation=cv2.INTER_AREA) if g.shape[0] != h0 else g


def _elastic(g, L, rng):
    amp = rng.uniform(0, MAX["elastic"]["amp"][L]); sig = MAX["elastic"]["sigma"]
    h, w = g.shape
    f = [cv2.GaussianBlur(rng.standard_normal((h, w)).astype(np.float32), (0, 0), sig) for _ in range(2)]
    dx, dy = (f[i] * (amp / max(1e-6, float(f[i].std()))) for i in (0, 1))
    xs, ys = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    return cv2.remap(g, xs + dx, ys + dy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=255)


def _stroke(g, L, rng):
    k = MAX["stroke"]["k"][L]; r = int(rng.integers(-1, k + 1))
    if r == 0: return g
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (abs(r) + 1, abs(r) + 1))
    return cv2.erode(g, ker) if r > 0 else cv2.dilate(g, ker)   # white background: erode = bolder ink, dilate = thinner ink


def _photo(g, L, rng):
    P = MAX["photo"]; x = g.astype(np.float32)
    b = rng.uniform(0, P["blur"][L])
    if b > 0.05: x = cv2.GaussianBlur(x, (0, 0), b)
    c = rng.uniform(0, P["contrast"][L]); x = 255 - (255 - x) * (1 - c)
    n = rng.uniform(0, P["noise"][L])
    if n > 0: x = x + rng.standard_normal(x.shape).astype(np.float32) * (255 * n)
    return np.clip(x, 0, 255).astype(np.uint8)


_FN = {"geom": _geom, "elastic": _elastic, "stroke": _stroke, "photo": _photo}


def augment(gray, family, level, rng):
    """gray: uint8 H x W (64 px high, white background). Returns a uint8 image of the same height (geom may widen it)."""
    if level == 0 or family in (None, "none"): return gray
    return _FN[family](gray, level, rng)


def to_line(image, height=64, max_width=None):
    """PIL image -> grayscale uint8 array, `height` px high (aspect kept; optionally squeezed to max_width)."""
    g = np.array(image.convert("L")); h, w = g.shape
    nw = max(1, round(w * height / h))
    if max_width and nw > max_width: nw = max_width
    return cv2.resize(g, (nw, height), interpolation=cv2.INTER_AREA if h > height else cv2.INTER_CUBIC)


In [ ]:
%%writefile metrics_hmt.py
# -*- coding: utf-8 -*-
"""Training-free metrics of every wave-1 augmentation condition, against the dev reference.

For each condition (none + 4 families x 4 levels) and draw k (K draws; `none` has no randomness -> 1 draw) the whole
training set (Option-B train minus the 20% dev writers, exactly the lines the recognizers were trained on) is augmented
once with augment.py and embedded; every metric is computed against the un-augmented dev lines (the reference).

Same protocol as the paper scorer (scorer-ocrs-dataset / paper_v3/analysis/score_all.py), code ported unchanged:
  granularities  patch (Otsu, 64 px, <=3 square windows/line), word (gap segmentation, gap calibrated on 300 train lines),
                 line (ink crop, pad to square), vit_token (TrOCR-base encoder tokens on the 384 tiled container)
  encoder        DINOv2-small (vit_small_patch14_dinov2.lvd142m, 224) for patch/word/line; TrOCR-base-handwritten for vit_token
  metrics        coverage, density, precision (k=5), Vendi + mean pairwise distance (2000-vector subsample),
                 Vendi SD over 10 subsamples, proxy A-distance (5-fold logistic regression), KID x1000 (10 x 1000)
  caps           every candidate granularity capped at 6000 vectors (seed 42); vit_token draws 6000 tokens from all lines

One JSON per (condition, draw) is pushed to HF dataset OUT_REPO/per/; a run skips files already there (resumable).
Usage: python metrics_hmt.py --shard i --nshards n [--draws 3] [--smoke]
"""
import argparse, io, json, os, sys, time

import numpy as np
import torch
from PIL import Image, ImageOps

import augment as A

OUT_REPO = "cyttic/hmt-metrics"
SEED, M_CAP, TOKEN_CAP, CHUNK_LINES, N_CALIB, ROW_H = 42, 6000, 6000, 100, 300, 64
DINO = "vit_small_patch14_dinov2.lvd142m"
TROCR = "microsoft/trocr-base-handwritten"
DEV = "cuda:0"


# ---------------------------------------------------------------- preprocessing (compare_datasets / common.py)
def otsu_binarize(img):
    a = np.asarray(img)
    hist = np.bincount(a.ravel(), minlength=256).astype(float)
    w = hist.cumsum(); m = (hist * np.arange(256)).cumsum()
    w1 = w[-1] - w; mu0 = np.where(w > 0, m / np.maximum(w, 1e-9), 0)
    mu1 = np.where(w1 > 0, (m[-1] - m) / np.maximum(w1, 1e-9), 0)
    t = int(np.argmax(w * w1 * (mu0 - mu1) ** 2))
    return Image.fromarray(np.where(a <= t, 0, 255).astype(np.uint8))


def patches(img, side, max_p=3, min_ink=0.01):
    a = np.asarray(img); ys, xs = np.where(a < 200)
    if len(ys) < 10: return []
    m = 4
    img = img.crop((max(0, xs.min() - m), max(0, ys.min() - m), min(img.width, xs.max() + 1 + m), min(img.height, ys.max() + 1 + m)))
    w, h = img.size; nw = max(side, int(w * side / h)); img = img.resize((nw, side), Image.LANCZOS)
    xs = [0] if nw <= side else list(np.linspace(0, nw - side, min(max_p, 1 + nw // side)).astype(int))
    res = []
    for x in xs:
        p = img.crop((x, 0, x + side, side))
        if (np.asarray(p) < 200).mean() >= min_ink: res.append(p.convert("RGB"))
    return res


def resize_to_h(img, h=64):
    w0, h0 = img.size
    return img if h0 == h else img.resize((max(1, int(w0 * h / h0)), h), Image.LANCZOS)


def ink_crop(img, margin=4):
    a = np.asarray(img); ys, xs = np.where(a < 200)
    if len(ys) < 10: return None
    return img.crop((max(0, xs.min() - margin), max(0, ys.min() - margin), min(img.width, xs.max() + 1 + margin), min(img.height, ys.max() + 1 + margin)))


def pad_to_square_resize(img, target=224, bg=255):
    w, h = img.size; s = target / max(w, h); nw, nh = max(1, int(w * s)), max(1, int(h * s))
    c = Image.new("L", (target, target), bg); c.paste(img.resize((nw, nh), Image.LANCZOS), ((target - nw) // 2, (target - nh) // 2))
    return c


def resize_h64_pad(img, target_h=64, canvas=224, bg=255):
    w, h = img.size; nw = max(1, int(w * target_h / h)); r = img.resize((nw, target_h), Image.LANCZOS)
    out = Image.new("L", (canvas, canvas), bg); x0 = max(0, (canvas - nw) // 2)
    if nw > canvas:
        cx0 = (nw - canvas) // 2; r = r.crop((cx0, 0, cx0 + canvas, target_h)); x0 = 0
    out.paste(r, (x0, (canvas - target_h) // 2))
    return out


def segment_words(img, gap_thresh):
    b = otsu_binarize(resize_to_h(img, 64)); ink = (np.asarray(b) < 128).any(axis=0)
    segs, in_word, gap, start, W = [], False, 0, 0, len(ink)
    for x in range(W):
        if ink[x]:
            if not in_word: start, in_word = x, True
            gap = 0
        elif in_word:
            gap += 1
            if gap > gap_thresh: segs.append((start, x - gap + 1)); in_word, gap = False, 0
    if in_word: segs.append((start, W - gap if gap else W))
    return [(s, e) for s, e in segs if e > s], b


def crop_word(b, x0, x1, margin=3):
    a = np.asarray(b); ys = np.where((a[:, max(0, x0 - margin):min(a.shape[1], x1 + margin)] < 128).any(axis=1))[0]
    y0, y1 = (0, a.shape[0]) if len(ys) == 0 else (max(0, ys.min() - margin), min(a.shape[0], ys.max() + 1 + margin))
    return b.crop((max(0, x0 - margin), y0, min(a.shape[1], x1 + margin), y1))


def calibrate_gap(lines):
    med_words = np.median([len(t.split()) for _, t in lines]); best = None
    for g in range(1, 41):
        d = abs(np.median([len(segment_words(im, g)[0]) for im, _ in lines]) - med_words)
        if best is None or d < best[0]: best = (d, g)
    return best[1]


# ---------------------------------------------------------------- embedders
class Dino:
    def __init__(self):
        import timm
        from timm.data import resolve_data_config, create_transform
        self.model = timm.create_model(DINO, pretrained=True, num_classes=0, img_size=224).to(DEV).eval()
        cfg = resolve_data_config({}, model=self.model); cfg["input_size"] = (3, 224, 224)
        self.tf, self.side = create_transform(**cfg), 224

    @torch.no_grad()
    def __call__(self, imgs, bs=64):
        feats = []
        for s in range(0, len(imgs), bs):
            x = torch.stack([self.tf(im.convert("RGB")) for im in imgs[s:s + bs]]).to(DEV)
            feats.append(self.model(x).float().cpu().numpy())
        E = np.concatenate(feats) if feats else np.zeros((0, 384), np.float32)
        return E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)


def tile384(img):
    """HebrewBlockProcessor with AUTOCROP off: mirror (RTL->LTR), 64 px high, tiled row by row into 384x384, [-1, 1]."""
    img = ImageOps.mirror(img.convert("RGB")); w, h = img.size; nw = max(1, round(w * 64 / h))
    a = np.array(img.resize((nw, 64), Image.LANCZOS)); c = Image.new("RGB", (384, 384), (255, 255, 255))
    sx = dx = dy = 0
    while sx < nw and dy < 384:
        cw = min(nw - sx, 384 - dx); c.paste(Image.fromarray(a[:, sx:sx + cw]), (dx, dy)); sx += cw; dx += cw
        if dx >= 384: dx, dy = 0, dy + 64
    return (torch.tensor(np.array(c), dtype=torch.float32).permute(2, 0, 1) / 255.0 - 0.5) / 0.5


class TrocrTokens:
    def __init__(self):
        from transformers import VisionEncoderDecoderModel
        self.enc = VisionEncoderDecoderModel.from_pretrained(TROCR).encoder.to(DEV).eval()

    @torch.no_grad()
    def __call__(self, imgs, bs=16):
        feats = []
        for s in range(0, len(imgs), bs):
            tok = self.enc(torch.stack([tile384(im) for im in imgs[s:s + bs]]).to(DEV)).last_hidden_state[:, 1:, :]
            feats.append(tok.reshape(-1, tok.shape[-1]).float().cpu().numpy())
        E = np.concatenate(feats) if feats else np.zeros((0, 768), np.float32)
        return E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)


def token_sample(vit, imgs, seed=SEED):
    rng = np.random.default_rng(seed); nch = -(-len(imgs) // CHUNK_LINES); per = -(-TOKEN_CAP // nch); out = []
    for c in range(nch):
        e = vit(imgs[c * CHUNK_LINES:(c + 1) * CHUNK_LINES]); out.append(e[rng.choice(len(e), min(per, len(e)), replace=False)])
    X = np.concatenate(out)
    return X[rng.choice(len(X), TOKEN_CAP, replace=False)] if len(X) > TOKEN_CAP else X


def cap(E, m=M_CAP, seed=SEED):
    return E if len(E) <= m else E[np.random.default_rng(seed).choice(len(E), m, replace=False)]


def embed_all(lines, dino, vit, gap):
    """lines: list of (PIL L image, text). Returns {granularity: vectors}, word retention."""
    imgs = [im for im, _ in lines]
    P = [p for im in imgs for p in patches(otsu_binarize(resize_to_h(im, 64)), dino.side)]
    W, kept = [], 0
    for im, t in lines:
        segs, b = segment_words(im, gap)
        if len(segs) != len(t.split()): continue
        crops = [crop_word(b, x0, x1) for x0, x1 in segs]
        if any(c.width < 2 or c.height < 2 for c in crops): continue
        kept += 1; W += [resize_h64_pad(c) for c in crops]
    L = [pad_to_square_resize(c) for c in (ink_crop(otsu_binarize(im)) for im in imgs) if c is not None]
    X = {"patch": cap(dino(P)), "word": cap(dino(W)), "line": cap(dino(L)), "vit_token": token_sample(vit, imgs)}
    return X, kept / len(lines)


# ---------------------------------------------------------------- metrics (compare_datasets.py, unchanged)
def kid(X, Y, subset=1000, reps=10, seed=0):
    d = X.shape[1]; rng = np.random.default_rng(seed); vals = []
    k = lambda a, b: (a @ b.T / d + 1.0) ** 3
    for _ in range(reps):
        x = X[rng.choice(len(X), min(subset, len(X)), replace=False)]; y = Y[rng.choice(len(Y), min(subset, len(Y)), replace=False)]
        m, n = len(x), len(y); kxx, kyy, kxy = k(x, x), k(y, y), k(x, y); np.fill_diagonal(kxx, 0); np.fill_diagonal(kyy, 0)
        vals.append(kxx.sum() / (m * (m - 1)) + kyy.sum() / (n * (n - 1)) - 2 * kxy.mean())
    return float(np.mean(vals)) * 1000, float(np.std(vals)) * 1000


def coverage_density(X, R, k=5):
    from sklearn.neighbors import NearestNeighbors
    from sklearn.metrics import pairwise_distances
    radii = NearestNeighbors(n_neighbors=k + 1).fit(R).kneighbors(R)[0][:, -1]
    d = pairwise_distances(R, X); within = d < radii[:, None]
    return float((d.min(axis=1) < radii).mean()), float(within.sum(axis=0).mean() / k), float(within.any(axis=0).mean())


def vendi(X, cap_=2000, seed=0):
    x = X[np.random.default_rng(seed).choice(len(X), min(cap_, len(X)), replace=False)]
    ev = np.clip(np.linalg.eigvalsh(x @ x.T / len(x)), 1e-12, None); ev /= ev.sum()
    return float(np.exp(-(ev * np.log(ev)).sum()))


def mean_pairwise(X, cap_=2000, seed=0):
    x = X[np.random.default_rng(seed).choice(len(X), min(cap_, len(X)), replace=False)]
    S = x @ x.T; n = len(x)
    return float(1 - (S.sum() - np.trace(S)) / (n * (n - 1)))


def a_distance(X, R, seed=0):
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score
    n = min(len(X), len(R)); rng = np.random.default_rng(seed)
    Z = np.concatenate([X[rng.choice(len(X), n, replace=False)], R[rng.choice(len(R), n, replace=False)]])
    acc = cross_val_score(LogisticRegression(max_iter=2000), Z, np.r_[np.zeros(n), np.ones(n)], cv=5).mean()
    return float(2 * (2 * acc - 1))


def score(X, R):
    cov, dens, prec = coverage_density(X, R); km, ks = kid(X, R)
    return dict(Cover=cov, Dens=dens, Prec=prec, Vendi=vendi(X), Vendi_sd=float(np.std([vendi(X, seed=s) for s in range(10)])),
                PairD=mean_pairwise(X), A_dist=a_distance(X, R), KID=km, KID_sd=ks, n=int(len(X)))


# ---------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--shard", type=int, default=0); ap.add_argument("--nshards", type=int, default=1)
    ap.add_argument("--draws", type=int, default=3); ap.add_argument("--smoke", action="store_true")
    a = ap.parse_args()
    from datasets import load_dataset
    from huggingface_hub import HfApi, create_repo
    api = HfApi(); create_repo(OUT_REPO, repo_type="dataset", exist_ok=True)
    done = {f for f in api.list_repo_files(OUT_REPO, repo_type="dataset")}

    hhd = load_dataset("cyttic/hhd-optionb-split")["train"]
    tw = list(hhd["writer"]); wr = sorted(set(tw)); __import__("random").Random(42).shuffle(wr); dev_w = set(wr[:int(len(wr) * 0.20)])
    to = lambda ex: (Image.fromarray(A.to_line(ex["image"], ROW_H)), ex["text"])
    train = [to(hhd[i]) for i, w in enumerate(tw) if w not in dev_w]          # (64 px L image, text): what the models trained on
    dev = [to(hhd[i]) for i, w in enumerate(tw) if w in dev_w]
    if a.smoke: train, dev = train[:120], dev[:60]
    print(f"train {len(train)} lines | dev {len(dev)} lines (reference)", flush=True)

    conds = ["none"] + [f"{f}{l}" for f in A.FAMILIES for l in A.LEVELS[1:]]
    jobs = [(c, k) for c in conds for k in range(1 if c == "none" else a.draws)]
    mine = [j for i, j in enumerate(jobs) if i % a.nshards == a.shard and f"per/{j[0]}_k{j[1]}.json" not in done]
    print(f"shard {a.shard}/{a.nshards}: {len(mine)} of {len(jobs)} jobs to do", flush=True)
    if not mine: return

    dino, vit = Dino(), TrocrTokens()
    gap = calibrate_gap(train[:N_CALIB])
    t0 = time.time(); R, ret_dev = embed_all(dev, dino, vit, gap)
    print(f"gap {gap} | reference sizes { {g: len(v) for g, v in R.items()} } | {time.time() - t0:.0f}s", flush=True)

    for c, k in mine:
        t0 = time.time(); fam, lvl = A.parse_condition(c); rng = np.random.default_rng([SEED, 1000 + k, conds.index(c)])
        cand = [(Image.fromarray(A.augment(np.asarray(im), fam, lvl, rng)), t) for im, t in train]
        X, ret = embed_all(cand, dino, vit, gap)
        res = dict(condition=c, family=fam or "none", level=lvl, draw=k, n_lines=len(cand), word_retention=ret,
                   dev_word_retention=ret_dev, gap_thresh=int(gap), protocol=dict(encoder=DINO, vit_token=TROCR, M_CAP=M_CAP,
                   TOKEN_CAP=TOKEN_CAP, seed=SEED, reference="dev writers (20% of Option-B train, seed 42), un-augmented"),
                   **{g: score(X[g], R[g]) for g in ("patch", "word", "line", "vit_token")})
        os.makedirs("per", exist_ok=True); json.dump(res, open(f"per/{c}_k{k}.json", "w"), indent=1)   # local copy too
        for attempt in range(5 if not a.smoke else 0):                    # two shards commit to one repo + flaky network: retry
            try:
                api.upload_file(path_or_fileobj=f"per/{c}_k{k}.json", path_in_repo=f"per/{c}_k{k}.json",
                                repo_id=OUT_REPO, repo_type="dataset", commit_message=f"{c} draw {k}"); break
            except Exception as e:
                print(f"upload retry {attempt + 1}: {e}", flush=True); time.sleep(30)
        print(f"[{c} k{k}] {time.time() - t0:.0f}s  Vendi p/w/l/t " + " ".join(f"{res[g]['Vendi']:.4f}" for g in ("patch", "word", "line", "vit_token"))
              + f"  KID line {res['line']['KID']:.4f}  ret {ret:.4f}", flush=True)


if __name__ == "__main__":
    main()


## Run: one worker per GPU (logs in `worker<i>.log`)

In [ ]:
import subprocess, os, time
DRAWS = 3
procs = [subprocess.Popen(f'python metrics_hmt.py --shard {i} --nshards 2 --draws {DRAWS} > worker{i}.log 2>&1', shell=True,
                          env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(i)}) for i in range(2)]
seen = [0, 0]
while any(p.poll() is None for p in procs):
    time.sleep(60)
    for i in range(2):
        lines = [l for l in open(f'worker{i}.log', errors='ignore').read().splitlines() if l.startswith(('[', 'train', 'shard', 'gap', 'upload'))]
        for l in lines[seen[i]:]: print(f'w{i}', l)
        seen[i] = len(lines)
print('exit codes', [p.returncode for p in procs])
for i in range(2):
    if procs[i].returncode: print(f'--- worker{i}.log tail ---'); print(open(f'worker{i}.log').read()[-3000:])

## Merge -> `metrics.json` + `metrics.csv` (one row per condition, draw, granularity)

In [ ]:
import json, glob, pandas as pd
from huggingface_hub import HfApi, snapshot_download
d = snapshot_download('cyttic/hmt-metrics', repo_type='dataset', allow_patterns=['per/*.json'])
runs = [json.load(open(f)) for f in sorted(glob.glob(f'{d}/per/*.json'))]
print(len(runs), 'of 49 jobs present')
rows = [dict(condition=r['condition'], family=r['family'], level=r['level'], draw=r['draw'], granularity=g,
             word_retention=r['word_retention'], **r[g]) for r in runs for g in ('patch', 'word', 'line', 'vit_token')]
df = pd.DataFrame(rows); df.to_csv('metrics.csv', index=False); json.dump(runs, open('metrics.json', 'w'), indent=1)
api = HfApi()
for f in ('metrics.csv', 'metrics.json'):
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id='cyttic/hmt-metrics', repo_type='dataset', commit_message='merge')
df.groupby(['granularity', 'condition'])[['Vendi', 'Vendi_sd', 'KID', 'Cover', 'Prec', 'A_dist']].mean().round(4)